# Terrain Wind Probe

In [1]:
from __future__ import annotations

import argparse
import json
import math
from dataclasses import asdict, dataclass
import os
import sys
from pathlib import Path

import requests
from dotenv import load_dotenv

def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for path in (current, *current.parents):
        if (path / "pyproject.toml").exists() and (path / "climbing_performance").exists():
            return path
    return current


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
load_dotenv(REPO_ROOT / ".env")

OPENTOPOGRAPHY_API_URL = "https://portal.opentopography.org/API/globaldem"
OPENTOPOGRAPHY_API_KEY_ENV_NAMES = ("OT_API_KEY", "OPENTOPOGRAPHY_API_KEY")
EARTH_RADIUS_M = 6_371_000.0

@dataclass(frozen=True)
class SamplePoint:
    bearing_deg: float
    distance_m: float
    latitude: float
    longitude: float
    elevation_m: float
    horizon_angle_deg: float


@dataclass(frozen=True)
class TerrainWindProbe:
    latitude: float
    longitude: float
    wind_from_deg: float
    center_elevation_m: float
    max_horizon_angle_deg: float
    mean_horizon_angle_deg: float
    shelter_index: float
    exposure_factor: float
    classification: str
    sample_count: int
    samples: list[SamplePoint]


def main() -> None:
    parser = argparse.ArgumentParser(
        description=(
            "Probe DEM-based terrain shelter for one coordinate and meteorological "
            "wind direction."
        )
    )
    parser.add_argument("--lat", type=float, required=True, help="Latitude in WGS84.")
    parser.add_argument("--lon", type=float, required=True, help="Longitude in WGS84.")
    parser.add_argument(
        "--wind-direction-deg",
        type=float,
        required=True,
        help="Meteorological wind direction: degrees wind is coming from.",
    )
    parser.add_argument("--radius-m", type=float, default=900.0)
    parser.add_argument("--step-m", type=float, default=90.0)
    parser.add_argument("--sector-deg", type=float, default=60.0)
    parser.add_argument("--rays", type=int, default=7)
    parser.add_argument("--demtype", default="COP30")
    parser.add_argument(
        "--json",
        action="store_true",
        help="Print the full probe result as JSON.",
    )
    args = parser.parse_args()

    probe = probe_terrain_wind(
        latitude=args.lat,
        longitude=args.lon,
        wind_from_deg=args.wind_direction_deg,
        radius_m=args.radius_m,
        step_m=args.step_m,
        sector_deg=args.sector_deg,
        ray_count=args.rays,
        demtype=args.demtype,
    )

    if args.json:
        print(json.dumps(probe_to_payload(probe), indent=2))
    else:
        print_summary(probe)


def probe_terrain_wind(
    *,
    latitude: float,
    longitude: float,
    wind_from_deg: float,
    radius_m: float = 900.0,
    step_m: float = 90.0,
    sector_deg: float = 60.0,
    ray_count: int = 7,
    demtype: str = "COP30",
) -> TerrainWindProbe:
    validate_inputs(latitude, longitude, radius_m, step_m, sector_deg, ray_count)

    sample_locations = [(latitude, longitude)]
    for bearing in wind_sector_bearings(wind_from_deg, sector_deg, ray_count):
        for distance in distances(step_m, radius_m):
            sample_locations.append(destination_point(latitude, longitude, bearing, distance))

    elevations = fetch_dem_elevations(sample_locations, demtype=demtype)
    center_elevation = elevations[0]
    samples = []
    index = 1
    for bearing in wind_sector_bearings(wind_from_deg, sector_deg, ray_count):
        for distance in distances(step_m, radius_m):
            sample_lat, sample_lon = sample_locations[index]
            sample_elevation = elevations[index]
            horizon_angle = math.degrees(
                math.atan2(sample_elevation - center_elevation, distance)
            )
            samples.append(
                SamplePoint(
                    bearing_deg=normalize_degrees(bearing),
                    distance_m=distance,
                    latitude=sample_lat,
                    longitude=sample_lon,
                    elevation_m=sample_elevation,
                    horizon_angle_deg=horizon_angle,
                )
            )
            index += 1

    max_horizon = max(sample.horizon_angle_deg for sample in samples)
    mean_horizon = sum(sample.horizon_angle_deg for sample in samples) / len(samples)
    shelter_index = max(0.0, max_horizon) + max(0.0, mean_horizon) * 0.5
    exposure_factor = shelter_to_exposure_factor(shelter_index)

    return TerrainWindProbe(
        latitude=latitude,
        longitude=longitude,
        wind_from_deg=normalize_degrees(wind_from_deg),
        center_elevation_m=center_elevation,
        max_horizon_angle_deg=max_horizon,
        mean_horizon_angle_deg=mean_horizon,
        shelter_index=shelter_index,
        exposure_factor=exposure_factor,
        classification=classify_exposure(exposure_factor),
        sample_count=len(samples),
        samples=samples,
    )


def validate_inputs(
    latitude: float,
    longitude: float,
    radius_m: float,
    step_m: float,
    sector_deg: float,
    ray_count: int,
) -> None:
    if not -90.0 <= latitude <= 90.0:
        raise ValueError("--lat must be between -90 and 90.")
    if not -180.0 <= longitude <= 180.0:
        raise ValueError("--lon must be between -180 and 180.")
    if radius_m <= 0.0:
        raise ValueError("--radius-m must be positive.")
    if step_m <= 0.0:
        raise ValueError("--step-m must be positive.")
    if radius_m < step_m:
        raise ValueError("--radius-m must be greater than or equal to --step-m.")
    if not 0.0 <= sector_deg <= 180.0:
        raise ValueError("--sector-deg must be between 0 and 180.")
    if ray_count < 1:
        raise ValueError("--rays must be at least 1.")


def wind_sector_bearings(
    wind_from_deg: float,
    sector_deg: float,
    ray_count: int,
) -> list[float]:
    if ray_count == 1:
        return [normalize_degrees(wind_from_deg)]

    start = wind_from_deg - sector_deg / 2.0
    step = sector_deg / (ray_count - 1)
    return [normalize_degrees(start + step * index) for index in range(ray_count)]


def distances(step_m: float, radius_m: float) -> list[float]:
    count = int(math.floor(radius_m / step_m))
    output = [step_m * index for index in range(1, count + 1)]
    if output[-1] < radius_m:
        output.append(radius_m)
    return output


def destination_point(
    latitude: float,
    longitude: float,
    bearing_deg: float,
    distance_m: float,
) -> tuple[float, float]:
    angular_distance = distance_m / EARTH_RADIUS_M
    bearing = math.radians(bearing_deg)
    lat1 = math.radians(latitude)
    lon1 = math.radians(longitude)

    lat2 = math.asin(
        math.sin(lat1) * math.cos(angular_distance)
        + math.cos(lat1) * math.sin(angular_distance) * math.cos(bearing)
    )
    lon2 = lon1 + math.atan2(
        math.sin(bearing) * math.sin(angular_distance) * math.cos(lat1),
        math.cos(angular_distance) - math.sin(lat1) * math.sin(lat2),
    )

    return math.degrees(lat2), normalize_longitude(math.degrees(lon2))

def fetch_dem_elevation(
    latitude: float,
    longitude: float,
    demtype: str = "COP30",
    radius_m: float = 250.0,
) -> float:
    return fetch_dem_elevations(
        [(latitude, longitude)],
        demtype=demtype,
        bbox_padding_m=radius_m,
    )[0]


def fetch_dem_elevations(
    locations: list[tuple[float, float]],
    *,
    demtype: str = "COP30",
    bbox_padding_m: float = 120.0,
) -> list[float]:
    if not locations:
        return []

    api_key = opentopography_api_key()
    params = {
        **bbox_for_locations(locations, padding_m=bbox_padding_m),
        "demtype": demtype,
        "outputFormat": "GTiff",
        "API_Key": api_key,
    }
    try:
        response = requests.get(OPENTOPOGRAPHY_API_URL, params=params, timeout=60)
        response.raise_for_status()
    except requests.RequestException as exc:
        raise RuntimeError(
            "OpenTopography DEM request failed. Check network access, DEM type, "
            "bounding box, and API key."
        ) from None
    if not response.content:
        raise RuntimeError(
            "OpenTopography returned an empty DEM response. "
            f"DEM type: {demtype}; bbox: {bbox_for_locations(locations, padding_m=bbox_padding_m)}. "
            "Try --demtype COP30 for global coverage or check that the selected DEM covers the area."
        )

    try:
        from rasterio.io import MemoryFile
    except ImportError as exc:
        raise RuntimeError(
            "rasterio is required to sample OpenTopography GeoTIFF responses. "
            "Install project dependencies with `python -m pip install -e .`."
        ) from exc

    try:
        with MemoryFile(response.content) as memory_file:
            with memory_file.open() as dataset:
                nodata = dataset.nodata
                elevations = []
                samples = dataset.sample(
                    [(lon, lat) for lat, lon in locations],
                    masked=True,
                )
                for sample in samples:
                    value = sample[0]
                    if getattr(value, "mask", False):
                        raise RuntimeError(
                            f"{demtype} has no DEM value for at least one sample point."
                        )
                    elevation = float(value)
                    if nodata is not None and math.isclose(elevation, float(nodata)):
                        raise RuntimeError(
                            f"{demtype} returned nodata for at least one sample point."
                        )
                    if elevation <= -9000.0:
                        raise RuntimeError(
                            f"{demtype} returned nodata-like elevation {elevation}."
                        )
                    elevations.append(elevation)
                return elevations
    except RuntimeError:
        raise
    except Exception as exc:
        raise RuntimeError(
            "Could not read OpenTopography response as GeoTIFF."
        ) from exc


def opentopography_api_key() -> str:
    for env_name in OPENTOPOGRAPHY_API_KEY_ENV_NAMES:
        api_key = os.getenv(env_name)
        if api_key:
            return api_key
    names = " or ".join(OPENTOPOGRAPHY_API_KEY_ENV_NAMES)
    raise RuntimeError(f"Set {names} in your environment or .env file.")


def bbox_for_locations(
    locations: list[tuple[float, float]],
    *,
    padding_m: float,
) -> dict[str, str]:
    min_lat = min(lat for lat, _ in locations)
    max_lat = max(lat for lat, _ in locations)
    min_lon = min(lon for _, lon in locations)
    max_lon = max(lon for _, lon in locations)
    mid_lat = (min_lat + max_lat) / 2.0
    lat_padding = padding_m / 111_320.0
    lon_padding = padding_m / (
        111_320.0 * max(0.01, math.cos(math.radians(mid_lat)))
    )
    return {
        "south": f"{min_lat - lat_padding:.6f}",
        "north": f"{max_lat + lat_padding:.6f}",
        "west": f"{min_lon - lon_padding:.6f}",
        "east": f"{max_lon + lon_padding:.6f}",
    }


def shelter_to_exposure_factor(shelter_index: float) -> float:
    return max(0.2, min(1.0, 1.0 - shelter_index / 18.0))


def classify_exposure(exposure_factor: float) -> str:
    if exposure_factor >= 0.8:
        return "exposed"
    if exposure_factor >= 0.55:
        return "partly sheltered"
    return "sheltered"


def normalize_degrees(value: float) -> float:
    return value % 360.0


def normalize_longitude(value: float) -> float:
    return ((value + 180.0) % 360.0) - 180.0


def probe_to_payload(probe: TerrainWindProbe) -> dict:
    payload = asdict(probe)
    payload["data_source"] = "OpenTopography Global DEM API"
    payload["interpretation"] = (
        "Experimental directional terrain shelter probe. Exposure factor is a "
        "screening value, not a calibrated wind-speed prediction."
    )
    return payload


def print_summary(probe: TerrainWindProbe) -> None:
    print("Terrain wind probe")
    print(f"Coordinate: {probe.latitude:.6f}, {probe.longitude:.6f}")
    print(f"Wind from: {probe.wind_from_deg:.0f} deg")
    print(f"Center elevation: {probe.center_elevation_m:.1f} m")
    print(f"Max upwind horizon: {probe.max_horizon_angle_deg:.2f} deg")
    print(f"Mean upwind horizon: {probe.mean_horizon_angle_deg:.2f} deg")
    print(f"Shelter index: {probe.shelter_index:.2f}")
    print(f"Exposure factor: {probe.exposure_factor:.2f}")
    print(f"Classification: {probe.classification}")
    print(f"Samples: {probe.sample_count}")



## Run A Probe

Set the coordinate and wind direction, then run the cell. This calls the OpenTopography API and requires `OT_API_KEY` or `OPENTOPOGRAPHY_API_KEY` in `.env` or the environment.

In [2]:
# Interpretation: lower exposure means the sampled upwind terrain rises enough to 
# plausibly shelter the coordinate from some ambient wind. Treat this as a first-pass 
# screening metric, not a calibrated local wind model."

# Hautacam, France: 42.972135312320056, -0.0075294177789186
latitude = 42.972135312320056
longitude = -0.0075294177789186

probe = probe_terrain_wind(
    latitude=latitude,
    longitude=longitude,
    wind_from_deg=21.0,
    radius_m=180.0,
    step_m=90.0,
    sector_deg=60.0,
    ray_count=7,
)
print_summary(probe)


Terrain wind probe
Coordinate: 42.972135, -0.007529
Wind from: 21 deg
Center elevation: 1518.6 m
Max upwind horizon: 20.43 deg
Mean upwind horizon: 15.25 deg
Shelter index: 28.05
Exposure factor: 0.20
Classification: sheltered
Samples: 14


# GPX sample test

In [3]:
from climbing_performance.gpx import parse_gpx, extract_climbs
from climbing_performance.weather import fetch_hourly_weather, nearest_weather_sample

route = parse_gpx(REPO_ROOT / "data" / "la_redoute.gpx")

if route.start_time is None or route.end_time is None:
    raise ValueError(
        "This GPX file has no timestamps. "
        "Use assign_estimated_times() before fetching time-specific weather."
    )

print(f"Distance: {route.distance_m / 1000:.2f} km")
print(f"Ascent: {route.ascent_m:.0f} m")

if route.min_elevation_m is not None and route.max_elevation_m is not None:
    print(
        f"Elevation range: "
        f"{route.min_elevation_m:.0f}–{route.max_elevation_m:.0f} m"
    )

climbs = extract_climbs(route)

weather_window = route.weather_query_window(
    location="midpoint",
)

samples = fetch_hourly_weather(
    latitude=weather_window.latitude,
    longitude=weather_window.longitude,
    start_date=weather_window.start_date,
    end_date=weather_window.end_date,
)

sample = nearest_weather_sample(samples, route.start_time)

latitude_str, longitude_str = f"({weather_window.latitude:.2f}", f"{weather_window.longitude:.2f})"
print(f"Nearest weather sample to route start time at: lat, lon: {latitude_str}, {longitude_str}")
print(sample)

Distance: 1.50 km
Ascent: 153 m
Elevation range: 136–289 m
Nearest weather sample to route start time at: lat, lon: (50.49, 5.71)
WeatherSample(time=datetime.datetime(2026, 4, 26, 17, 0, tzinfo=datetime.timezone(datetime.timedelta(seconds=7200))), temperature_c=15.4, relative_humidity_pct=51.0, wind_speed_m_s=3.28, wind_direction_deg=21.0, wind_gusts_m_s=7.6)


In [ ]:
if route.start_time is None or route.end_time is None:
    raise ValueError(...)

# Weather sample test

In [4]:
from datetime import datetime

from climbing_performance.weather import (
    fetch_hourly_weather,
    nearest_weather_sample,
    wind_to_components,
)


samples = fetch_hourly_weather(
    latitude=latitude,
    longitude=longitude,
    start_date=datetime.today().date().isoformat(),
    end_date=datetime.today().date().isoformat(),
)

target = datetime.now(samples[0].time.tzinfo)

weather = nearest_weather_sample(samples, target)

# Suppose rider heading is 210° true bearing
headwind, crosswind = wind_to_components(
    wind_speed_m_s=weather.wind_speed_m_s,
    wind_direction_deg=weather.wind_direction_deg,
    rider_heading_deg=210.0,
)

print(f"Nearest sample: {weather.time}")
print(f"Temperature: {weather.temperature_c:.1f} °C")
print(f"Relative humidity: {weather.relative_humidity_pct:.0f}%")
print(f"Wind speed: {weather.wind_speed_m_s:.2f} m/s")
print(f"Wind direction: {weather.wind_direction_deg:.0f}°")
print(f"Wind gusts: {weather.wind_gusts_m_s:.2f} m/s" if weather.wind_gusts_m_s is not None else "Wind gusts: unavailable")
print(f"Headwind component: {headwind:.2f} m/s")
print(f"Crosswind component: {crosswind:.2f} m/s")

Nearest sample: 2026-05-19 00:00:00+02:00
Temperature: 3.9 °C
Relative humidity: 89%
Wind speed: 0.45 m/s
Wind direction: 117°
Wind gusts: 1.00 m/s
Headwind component: -0.02 m/s
Crosswind component: 0.45 m/s
